# Cu-64 nel nucleo: microdosimetria → MKM → LQ → S(N) e RBE(N)

Metti questo notebook nella **radice del progetto** (dove c'è la cartella `runs/`) ed esegui le celle in ordine.
Il passo lento è uno solo (la cella *"Campionamento delle dosi"*): il risultato viene salvato in `cache_dosi.npz`,
quindi puoi cambiare pesi, `alpha_0`, `beta`, `G` e rifare grafici e tabelle in pochi secondi.

## 0. Parametri

In [ ]:
import glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = sorted(glob.glob("runs/run_*/yz.root"))
TREE = "yz"

# --- modello ---
R_d, R_n = 0.420, 4.0        # um   raggio dominio (= /microyz/det/Radius) e raggio nucleo
y0 = 150.0                   # keV/um  saturazione MKM
alpha_0, beta = 0.147, 0.02  # Gy^-1, Gy^-2 (raggi X)
G = 1.0                      # Lea-Catcheside: 1 = dose istantanea, <1 se protratta
rho = 1e-15                  # kg/um^3
KEV_J, EV_J = 1.602176634e-16, 1.602176634e-19
S_TARGET = 0.10
K_CONV = KEV_J / (rho * np.pi * R_d**2)     # Gy per keV/um

# --- statistica ---
N_SIM = 400                  # cellule virtuali per pool
N_POINTS = 60                # punti sull'asse N
N_BOOT = 20                  # pool bootstrap (solo se hai UN file)
SEED = 12345
WEIGHT_MODES = ("1/n", "N/n")   # 1/n: correzione dentro l'evento; N/n: anche tra eventi
PLOT_MODE = "N/n"

print(f"{len(FILES)} file trovati;  K_CONV = {K_CONV:.4f} Gy per keV/um")

## 1. Lettura dei file

In [ ]:
import uproot

cols = ["y", "nbHits", "nbScoredHits", "Nuclear_dose"]
dfs = [uproot.open(f)[TREE].arrays(cols, library="pd") for f in FILES]
print("eventi per file:", [len(d) for d in dfs])
dfs[0].head()

## 2. Controlli
- **Allineamento:** l'energia nel nucleo (da `Nuclear_dose`) deve essere ≥ quella nel dominio (da `y`). Frazione attesa: 1.0000.
- **Seed:** i run devono avere eventi diversi.

In [ ]:
def check_alignment(df, label=""):
    m_nucl = 1e3 * 4/3 * np.pi * (R_n * 1e-6) ** 3
    e_nucl = df["Nuclear_dose"].to_numpy() * m_nucl / EV_J     # eV nel nucleo
    eps = df["y"].to_numpy() * (4 * R_d / 3 * 1000.0)          # eV nel dominio
    ok = np.mean(e_nucl >= eps * (1 - 1e-3) - 0.5)
    print(f"[check] {label}: righe coerenti = {ok:.4f}")
    return ok

for f, d in zip(FILES, dfs):
    check_alignment(d, f)

if len(dfs) > 1:
    heads = [d["y"].to_numpy()[:200].tobytes() for d in dfs]
    print("[seed] run tutti diversi" if len(set(heads)) == len(heads) else "[ATTENZIONE] run identici: seed non cambiati!")

## 3. Pool (un pool = una simulazione indipendente)

In [ ]:
def make_pools(dfs, rng):
    if len(dfs) > 1:
        return dfs
    df = dfs[0]
    print(f"un solo file: uso {N_BOOT} pool bootstrap")
    return [df.iloc[rng.integers(0, len(df), len(df))].reset_index(drop=True) for _ in range(N_BOOT)]

rng = np.random.default_rng(SEED)
pools = make_pools(dfs, rng)
print(len(pools), "pool")

## 4. Microdosimetria: y_F, y_D, y*
Per ogni pool e per ciascuno schema di peso. Gli errori (`sem`) sono tra pool.

In [ ]:
def weights(df, mode):
    n = df["nbScoredHits"].to_numpy(float)
    big_n = df["nbHits"].to_numpy(float)
    return 1.0 / n if mode == "1/n" else big_n / n

def micro(df, mode):
    y = df["y"].to_numpy()
    w = weights(df, mode)
    y_f = np.sum(w * y) / np.sum(w)
    y_d = np.sum(w * y**2) / np.sum(w * y)
    y_star = np.sum(w * y0**2 * (1.0 - np.exp(-(y / y0) ** 2))) / np.sum(w * y)
    return y_f, y_d, y_star

rows = []
for i, p in enumerate(pools):
    for mode in WEIGHT_MODES:
        yf, yd, ys = micro(p, mode)
        rows.append(dict(pool=i, peso=mode, yF=yf, yD=yd, ystar=ys,
                         alpha=alpha_0 + beta * K_CONV * ys))
tab_micro = pd.DataFrame(rows)
tab_micro.groupby("peso")[["yF", "yD", "ystar", "alpha"]].agg(["mean", "sem"])

## 5. Spettri pesati f(y) e d(y) (tutti i run insieme)

In [ ]:
allp = pd.concat(pools, ignore_index=True)
bins = np.logspace(-2, np.log10(20), 60)
x = np.sqrt(bins[:-1] * bins[1:])
dln = np.diff(np.log(bins))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for mode in WEIGHT_MODES:
    w = weights(allp, mode)
    y = allp["y"].to_numpy()
    f = np.histogram(y, bins=bins, weights=w)[0];     f = f / f.sum() / dln
    d = np.histogram(y, bins=bins, weights=w * y)[0]; d = d / d.sum() / dln
    ax[0].semilogx(x, f, label=f"peso {mode}")
    ax[1].semilogx(x, d, label=f"peso {mode}")
ax[0].set_ylabel("y f(y)"); ax[1].set_ylabel("y d(y)")
for a in ax:
    a.set_xlabel("y [keV/um]"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.show()

## 6. Campionamento delle dosi (passo lento, una volta sola)
Ogni cellula virtuale accumula decadimenti uno dopo l'altro; la dose a N decadimenti è la somma cumulata delle dosi nucleari.
Una sola estrazione per cellula serve per tutti gli N.

In [ ]:
def d_ref_at(s_target):
    """Dose X acuta (alpha_0, beta) per avere S = s_target."""
    return (-alpha_0 + np.sqrt(alpha_0**2 - 4 * beta * np.log(s_target))) / (2 * beta)

def sample_doses(dn, ns, rng):
    n_max = int(ns.max())
    idx = rng.integers(0, dn.size, size=(N_SIM, n_max), dtype=np.int32)
    cum = np.cumsum(dn[idx], axis=1)
    return cum[:, ns - 1].T                      # shape (len(ns), N_SIM)

dn_mean = np.mean([p["Nuclear_dose"].mean() for p in pools])
n_max = int(3 * d_ref_at(S_TARGET) / dn_mean)
ns = np.unique(np.linspace(1, n_max, N_POINTS).astype(int))
print(f"dose nucleare media per decadimento = {dn_mean:.3e} Gy ;  N da 1 a {n_max}")

t0 = time.time()
d_mats = []
for i, p in enumerate(pools):
    d_mats.append(sample_doses(p["Nuclear_dose"].to_numpy(), ns, rng))
    print(f"pool {i + 1}/{len(pools)}  ({time.time() - t0:.0f} s)")
d_mats = np.array(d_mats)                       # (pool, N, cellule)
np.savez("cache_dosi.npz", ns=ns, d_mats=d_mats)

*Solo se riavvii il kernel:* non rifare il campionamento, ricarica la cache (e rilancia le celle 0-4).

In [ ]:
# cache = np.load("cache_dosi.npz"); ns = cache["ns"]; d_mats = cache["d_mats"]

## 7. LQ + MKM: S(N), RBE(N), N10, D10 (veloce)

In [ ]:
def lq(d_mat, alpha, g=None):
    g = G if g is None else g
    e = alpha * d_mat + beta * g * d_mat**2
    s = np.exp(-e)
    d_ref = (-alpha_0 + np.sqrt(alpha_0**2 + 4 * beta * e)) / (2 * beta)
    rbe = np.where(d_mat > 0, d_ref / np.where(d_mat > 0, d_mat, 1.0), np.nan)
    return s, rbe

def crossing(ns, s_mean, d_mean, target=S_TARGET):
    if s_mean.min() > target:
        return np.nan, np.nan                    # S non arriva al target nella griglia
    s = np.maximum(np.minimum.accumulate(s_mean), 1e-300)
    ls = np.log(s)
    return (np.interp(np.log(target), ls[::-1], ns[::-1]),
            np.interp(np.log(target), ls[::-1], d_mean[::-1]))

def analyze(df, d_mat, mode, g=None):
    y_f, y_d, y_star = micro(df, mode)
    alpha = alpha_0 + beta * K_CONV * y_star
    s, rbe = lq(d_mat, alpha, g)
    s_mean, d_mean = s.mean(axis=1), d_mat.mean(axis=1)
    n10, d10 = crossing(ns, s_mean, d_mean)
    return dict(S=s_mean, S_cell=s.std(axis=1),
                rbe=np.nanmean(rbe, axis=1), rbe_cell=np.nanstd(rbe, axis=1), D=d_mean,
                yF=y_f, yD=y_d, ystar=y_star, alpha=alpha,
                N10=n10, D10=d10, rbe10=d_ref_at(S_TARGET) / d10, rbe_max=alpha / alpha_0)

def aggregate(results):
    k = len(results)
    agg = {}
    for key in ("S", "rbe", "D"):
        arr = np.array([r[key] for r in results])
        agg[key] = arr.mean(axis=0)
        agg[key + "_sem"] = arr.std(axis=0, ddof=1) / np.sqrt(k)
    for key in ("S_cell", "rbe_cell"):
        agg[key] = np.mean([r[key] for r in results], axis=0)
    for key in ("yF", "yD", "ystar", "alpha", "N10", "D10", "rbe10", "rbe_max"):
        v = np.array([r[key] for r in results])
        agg[key] = v.mean()
        agg[key + "_sem"] = v.std(ddof=1) / np.sqrt(k)
    return agg

results = {m: [analyze(p, dm, m) for p, dm in zip(pools, d_mats)] for m in WEIGHT_MODES}
agg = {m: aggregate(results[m]) for m in WEIGHT_MODES}

tab = pd.DataFrame({
    m: {"y_F": a["yF"], "y_D": a["yD"], "y*": a["ystar"], "alpha": a["alpha"], "RBE(D->0)": a["rbe_max"],
        "N10": a["N10"], "N10 sem": a["N10_sem"], "D10 [Gy]": a["D10"], "D10 sem": a["D10_sem"],
        "RBE10": a["rbe10"], "RBE10 sem": a["rbe10_sem"]}
    for m, a in agg.items()}).T
print(f"Dose X di riferimento per S=10%: {d_ref_at(S_TARGET):.4f} Gy")
tab

## 8. Grafico: sopravvivenza e RBE

In [ ]:
a = agg[PLOT_MODE]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9), sharex=True)

ax1.plot(ns, a["S"], color="#1f77b4", label=r"$\langle S\rangle$ Cu-64 (MKM-LQ)")
ax1.fill_between(ns, np.clip(a["S"] - a["S_cell"], 1e-12, None), a["S"] + a["S_cell"],
                 color="#1f77b4", alpha=0.2, label="variabilità tra cellule (1σ)")
ax1.errorbar(ns, a["S"], yerr=a["S_sem"], fmt="none", ecolor="k", elinewidth=0.8, capsize=2,
             label="errore sulla media (tra simulazioni)")
ax1.plot(ns, np.exp(-alpha_0 * a["D"] - beta * a["D"] ** 2), "--", color="gray",
         label="raggi X acuti, stessa dose media")
ax1.axhline(S_TARGET, color="darkgreen", ls="--", lw=1.2)
ax1.axvline(a["N10"], color="darkgray", ls=":", lw=1.2, label=f"$N_{{10}}$ = {a['N10']:.0f} ± {a['N10_sem']:.0f}")
ax1.set_yscale("log"); ax1.set_ylim(1e-3, 1.5)
ax1.set_ylabel("Frazione di sopravvivenza S")
ax1.grid(True, which="both", alpha=0.3); ax1.legend(loc="lower left", fontsize=9)

ax2.plot(ns, a["rbe"], color="crimson", label=f"RBE (peso {PLOT_MODE})")
ax2.fill_between(ns, a["rbe"] - a["rbe_cell"], a["rbe"] + a["rbe_cell"], color="crimson", alpha=0.2)
ax2.errorbar(ns, a["rbe"], yerr=a["rbe_sem"], fmt="none", ecolor="k", elinewidth=0.8, capsize=2)
ax2.axhline(1.0, color="gray", ls="--", lw=1)
ax2.set_xlabel("Numero di decadimenti N"); ax2.set_ylabel("RBE")
ax2.grid(True, alpha=0.3); ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout(); plt.savefig("cu64_S_RBE.png", dpi=200); plt.show()

## 9. (Opzionale) Effetto della riparazione: scansione di G
Dose protratta in ore (emivita del Cu-64 ≈ 12.7 h) → il termine quadratico si riduce (`G < 1`).
Usa la cache delle dosi, quindi è istantaneo. Con `G` piccolo S può non arrivare al 10% nella griglia di N: in quel caso compare `NaN`.

In [ ]:
rows = []
for g in (1.0, 0.5, 0.2, 0.1, 0.05):
    res = [analyze(p, dm, PLOT_MODE, g=g) for p, dm in zip(pools, d_mats)]
    n10 = np.array([r["N10"] for r in res]); r10 = np.array([r["rbe10"] for r in res])
    rows.append(dict(G=g, N10=np.nanmean(n10), N10_sem=np.nanstd(n10, ddof=1) / np.sqrt(len(n10)),
                     RBE10=np.nanmean(r10)))
pd.DataFrame(rows)